# FIN510 Coursework 2 Option A | Factor replication

An investment committee is considering a value or momentum tilt. Choose one published JKP theme and one market. Estimate how much the theme earned after allowing for its exposure to the market. Then check whether the evidence looks similar in two parts of the sample before making a recommendation.

The starter uses January 1990 to December 2023, a window available for both markets and themes. Split it into two 17-year blocks, 1990–2006 and 2007–2023. Choose your theme and market before you see the estimates. Keep a record of any variants you try later.

In Colab, upload this `.ipynb` file and run the cells in order. The setup downloads a pinned copy of the course data. For a local run you can place that same CSV next to the notebook.

## How much return does the model attribute to market exposure?

Consider an illustrative value spread. Suppose a fitted market model gives it a market beta of 0.40. If the market excess return averages 0.50% per month and the spread averages 0.35% per month, the market contribution is $0.40\times0.50=0.20$ percentage points per month. That leaves a market-adjusted alpha of $0.35-0.20=0.15$ percentage points per month. Estimate both quantities from your chosen sample and check how precisely the data estimate alpha.

## Choose and inspect the return series

In the course extract, `HML` is the JKP value theme and `MOM` is the JKP momentum theme. These series combine published long–short factor returns within each theme. For this assignment, reproduce evidence for the chosen theme using its supplied returns. `MKT` is the excess market return for the same country, which we use as the benchmark. The file stores monthly USD returns in decimals. A value of `0.01` becomes 1% when we multiply by 100. [JKP data definitions](https://jkpfactors.com/data)

Compare the dates, means and standard deviations below. Use this description to set up your interpretation of the regression. [Pinned course extract](https://github.com/quinfer/fin510-colab-notebooks/blob/2263bb8e5fa692bbca38e9f0f766c592549f217a/labs/jkp_master_global_monthly.csv)

In [ ]:
import hashlib
from io import BytesIO
from pathlib import Path
import subprocess
import sys
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import statsmodels.api as sm
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels==0.14.5"])
    import statsmodels.api as sm

MARKET = "usa"  # Choose "usa" or "gbr".
FACTOR = "HML"  # Choose "HML" (value) or "MOM" (momentum).
markets = {"usa": "United States", "gbr": "United Kingdom"}
factors = {"HML": "Value", "MOM": "Momentum"}
if MARKET not in markets or FACTOR not in factors:
    raise ValueError('Use "usa" or "gbr" for MARKET and "HML" or "MOM" for FACTOR.')

data_url = (
    "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
    "2263bb8e5fa692bbca38e9f0f766c592549f217a/labs/"
    "jkp_master_global_monthly.csv"
)
expected_sha256 = "6a4eb3034551694041a149625777a222d3e3ac0a4e442b2b85f759347b28d24d"
local_data = Path("jkp_master_global_monthly.csv")
if local_data.is_file():
    payload = local_data.read_bytes()
else:
    with urlopen(data_url, timeout=45) as response:
        payload = response.read()
if hashlib.sha256(payload).hexdigest() != expected_sha256:
    raise ValueError("Use the pinned course CSV linked above.")

source = pd.read_csv(BytesIO(payload), usecols=["country", "date", FACTOR, "MKT"])
chosen = source.loc[source["country"].eq(MARKET)].copy()
chosen["month"] = pd.to_datetime(chosen["date"]).dt.to_period("M")
months = pd.period_range("1990-01", "2023-12", freq="M")
chosen = chosen.loc[chosen["month"].isin(months)]
if chosen["month"].duplicated().any():
    raise ValueError("The selected market has a duplicate month.")
returns = chosen.set_index("month")[[FACTOR, "MKT"]].reindex(months).mul(100)
if not np.isfinite(returns.to_numpy()).all():
    raise ValueError("The benchmark and factor must both cover every sample month.")

print(f"{markets[MARKET]} | {factors[FACTOR]} ({FACTOR}) | Benchmark {markets[MARKET]} MKT")
print(f"{returns.index[0]} to {returns.index[-1]} | {len(returns)} monthly observations")
performance = pd.DataFrame({
    "Mean (% per month)": returns.mean(),
    "SD (% per month)": returns.std(),
})
display(performance.round(4))

## Estimate market-adjusted alpha

Regress each month's factor return in per cent, $f_t$, on an intercept and the same month's market excess return in per cent, $m_t$.

$$
f_t=\alpha+\beta m_t+\varepsilon_t.
$$

The OLS intercept $\alpha$ estimates the factor return at zero market excess return, in percentage points per month. The slope $\beta$ estimates how much the factor changes when the market excess return changes by one percentage point.

Regression errors can vary in spread and be correlated across months. We use six monthly lags for the Newey–West heteroskedasticity and autocorrelation consistent (HAC) standard errors, as in Lab 9. The table's two-sided p-values and 95% intervals use a normal approximation. The HAC results use ordinary least squares for the coefficient estimates.

In [ ]:
HAC_LAGS = 6


def fit_market_model(sample):
    benchmark = sm.add_constant(sample[["MKT"]], has_constant="add")
    return sm.OLS(sample[FACTOR], benchmark).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": HAC_LAGS, "kernel": "bartlett"},
        use_t=False,
    )


full = fit_market_model(returns)
interval = full.conf_int(alpha=0.05)
coefficients = pd.DataFrame({
    "Estimate": full.params,
    "HAC SE": full.bse,
    "95% lower": interval[0],
    "95% upper": interval[1],
    "Two-sided p-value": full.pvalues,
}).rename(index={"const": "Alpha (pp per month)", "MKT": "Market beta"})

print(f"Newey–West, {HAC_LAGS} monthly lags, normal approximation")
display(coefficients.style.format({
    "Estimate": "{:.4f}", "HAC SE": "{:.4f}",
    "95% lower": "{:.4f}", "95% upper": "{:.4f}",
    "Two-sided p-value": "{:.3g}",
}))

## Check the two periods

Fit the same regression separately before and from January 2007, using the same benchmark and HAC settings. Compare the alpha estimates and their uncertainty. Each interval shows uncertainty about that period's own alpha. A change in the two p-values alone gives limited evidence about a change in alpha.

In [ ]:
results = []
for label, sample, model in [
    ("Full sample", returns, full),
    ("1990–2006", returns.loc[:"2006-12"], None),
    ("2007–2023", returns.loc["2007-01":], None),
]:
    if model is None:
        model = fit_market_model(sample)
    lower, upper = model.conf_int(alpha=0.05).loc["const"]
    results.append({
        "Period": label,
        "Start": str(sample.index[0]),
        "End": str(sample.index[-1]),
        "Months": len(sample),
        "Alpha (pp per month)": model.params["const"],
        "95% lower": lower,
        "95% upper": upper,
        "HAC p-value": model.pvalues["const"],
    })

robustness = pd.DataFrame(results).set_index("Period")
display(robustness.style.format({
    "Alpha (pp per month)": "{:.4f}", "95% lower": "{:.4f}",
    "95% upper": "{:.4f}", "HAC p-value": "{:.3g}",
}))

In [ ]:
estimate = robustness["Alpha (pp per month)"]
error = np.vstack([estimate - robustness["95% lower"], robustness["95% upper"] - estimate])
fig, ax = plt.subplots(figsize=(8, 3), layout="constrained")
ax.errorbar(estimate, robustness.index, xerr=error, fmt="o", capsize=4, color="tab:blue")
ax.axvline(0, linestyle="--", color="0.5", linewidth=1)
ax.invert_yaxis()
ax.set_xlabel("Market-adjusted alpha (percentage points per month)")
ax.set_title(f"{markets[MARKET]} {factors[FACTOR].lower()} theme · 95% HAC intervals")
plt.show()

## Make an investment decision

- Use the full-sample means and beta to reproduce the market-adjusted alpha, as in the worked example. Explain what the intercept and its interval tell you about the factor in your market and sample.
- Compare the two periods. How large is the change relative to the uncertainty about each estimate? Explain what your result can establish about the theme in the supplied data and how the sample or factor construction may differ from the published evidence. [Jensen, Kelly and Pedersen (2023)](https://doi.org/10.1111/jofi.13249)
- Recommend whether to consider a tilt and say what evidence would change your decision. Consider the size of the estimated return and the costs of trading or shorting. The file supplies returns without holdings or turnover. Label any cost figure you use as a scenario and state what extra data you would use to estimate it.

Follow the coursework brief for submission requirements. Cite the data and explain the sample, methods, results and limitations in your own words.